# Đánh giá Test Set: Baseline và Exp6 (Tiling) trên Kaggle

Notebook này thực hiện từ A-Z để đánh giá test set trên Kaggle:
1. Clone Git Repo (bỏ `.git`)
2. Copy dataset TACO từ input Kaggle
3. Chạy Data Preparation (Cleaning -> Convert -> Split)
4. Chạy Tiling để chuẩn bị data test cho Exp 6
5. Chạy Evaluate (`yolo val`) trên tập Test cho cả 2 model.
6. Nén và xuất kết quả ra file ZIP để tải về.

In [1]:
# 1. Clone Github Repository
!git clone https://github.com/Shiba-dotcom/waste-detection_project.git /kaggle/working/waste-detection2-Stage

# Xóa thư mục .git cho nhẹ bộ nhớ
!rm -rf /kaggle/working/waste-detection2-Stage/.git

# Cài đặt các thư viện cần thiết
%cd /kaggle/working/waste-detection2-Stage
!pip install -r requirements.txt

Cloning into '/kaggle/working/waste-detection2-Stage'...
remote: Enumerating objects: 3884, done.
remote: Counting objects: 100% (61/61), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 3884 (delta 21), reused 35 (delta 17), pack-reused 3823 (from 3)
Receiving objects: 100% (3884/3884), 2.84 GiB | 22.86 MiB/s, done.
Resolving deltas: 100% (226/226), done.
Updating files: 100% (161/161), done.
/kaggle/working/waste-detection2-Stage
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 41.3 MB/s eta 0:00:00


In [2]:
# ============================================================
# 2. Nhập Dữ liệu Ngoại lai (TACO)
# ============================================================
import os, shutil

# Chỉ tạo thư mục đích cho dữ liệu TACO
!mkdir -p /kaggle/working/waste-detection2-Stage/data/raw

datasets_to_copy = [
    {"src": "/kaggle/input/datasets/sohamchaudhari2004/taco-trash-detection-dataset/data",
     "dst": "/kaggle/working/waste-detection2-Stage/data/raw"}
]

for task in datasets_to_copy:
    if os.path.exists(task["src"]):
        os.makedirs(task["dst"], exist_ok=True)
        shutil.copytree(task["src"], task["dst"], dirs_exist_ok=True)
        print(f"Đã tải: {os.path.basename(task['src'])}")
    else:
        print(f"Bỏ qua: {task['src']} (Không tìm thấy trên Kaggle Dataset)")

Đã tải: data


In [3]:
# 3. Tiền xử lý dữ liệu: Cleaning -> YOLO Format -> Split Dataset
%cd /kaggle/working/waste-detection2-Stage/src/data_prep
!python data_cleaning.py

%cd /kaggle/working/waste-detection2-Stage/src
!python Training_dataYolo.py

%cd /kaggle/working/waste-detection2-Stage/src/data_prep
!python split_dataset.py

/kaggle/working/waste-detection2-Stage/src/data_prep
  DATA CLEANING - TACO Dataset

[Load] annotations.json ...
  So anh goc         : 1500
  So annotations goc : 4784
  So categories      : 60

────────────────────────────────────────────────────────────
[Buoc 1] Kiem tra dong trung lap (Duplicates)
────────────────────────────────────────────────────────────
  Duplicate annotations: 0
  Duplicate image IDs: 0
  Duplicate file names: 0

────────────────────────────────────────────────────────────
[Buoc 2] Kiem tra gia tri thieu (Missing Values)
────────────────────────────────────────────────────────────
  Images: Tat ca truong bat buoc day du
  Annotations: Tat ca truong bat buoc day du
  Anh khong co annotation: 0

────────────────────────────────────────────────────────────
[Buoc 3] Kiem tra nhan khong hop le
────────────────────────────────────────────────────────────
  Annotations voi category_id khong hop le: 0
  Categories khong co trong mapping.csv: 1
    - 'Rope & strings' -

In [4]:
# 4. Chuẩn bị tập dữ liệu Tiling (dùng cho đánh giá Exp 6)
%cd /kaggle/working/waste-detection2-Stage/src/data_prep
!python tiling.py

/kaggle/working/waste-detection2-Stage/src/data_prep
  TILING CONFIG
  TILE_SIZE        : 640 px
  OVERLAP          : 20%  (stride = 512 px)
  INCLUDE_EMPTY    : False
  MIN_BOX_AREA_RATIO: 0.1

[INFO] Tiling split: train  (apply_empty_cap=True)
  Ảnh gốc      :  1,048
  Tiles lưu    :  6,839  (6.5x mở rộng)
    ├─ có object:  6,839
    ├─ trống (lưu)   :      0  (0.0% của tổng tiles)
    └─ trống (bỏ qua): 32,688

[INFO] Tiling split: val  (apply_empty_cap=False)
  Ảnh gốc      :    224
  Tiles lưu    :  1,637  (7.3x mở rộng)
    ├─ có object:  1,637
    ├─ trống (lưu)   :      0  (0.0% của tổng tiles)
    └─ trống (bỏ qua):  6,783

[INFO] Tiling split: test  (apply_empty_cap=False)
  Ảnh gốc      :    228
  Tiles lưu    :  1,643  (7.2x mở rộng)
    ├─ có object:  1,643
    ├─ trống (lưu)   :      0  (0.0% của tổng tiles)
    └─ trống (bỏ qua):  7,210

  TỔNG: 1,500 ảnh gốc → 10,119 tiles
Done.


## 5. Đánh giá Mô Hình (Evaluation)


In [5]:
# Đánh giá Baseline trên tập Test gốc (Dataset chuẩn)
%cd /kaggle/working/waste-detection2-Stage

!yolo val model=results/runs/baseline_yolov8n/weights/best.pt data=data/processed/dataset.yaml split=test project=results/runs name=baseline_test_eval

/kaggle/working/waste-detection2-Stage
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.57 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
Model summary (fused): 73 layers, 3,006,623 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1235.9±201.9 MB/s, size: 2037.3 KB)
val: Scanning /kaggle/working/waste-detection2-Stage/data/processed/labels/test/batch_1... 228 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 228/228 831.8it/s 0.3s0.2s
val: New cache created: /kaggle/working/waste-detection2-Stage/data/processed/labels/test/batch_1.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 2.9s/it 

In [7]:
# Đánh giá Tiling Model (Exp6) trên tập Test đã được Tiling
%cd /kaggle/working/waste-detection2-Stage

!yolo val model=/kaggle/input/datasets/tunkitshi/exp6-tiling/best.pt data=data/processed_tiled/dataset.yaml split=test project=results/runs name=exp6_tiling_test_eval

/kaggle/working/waste-detection2-Stage
Ultralytics 8.4.57 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
Model summary (fused): 73 layers, 3,006,623 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1345.6±1016.1 MB/s, size: 120.7 KB)
val: Scanning /kaggle/working/waste-detection2-Stage/data/processed_tiled/labels/test/batch_1... 1643 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1643/1643 1.1Kit/s 1.6s0.0s
val: New cache created: /kaggle/working/waste-detection2-Stage/data/processed_tiled/labels/test/batch_1.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 103/103 3.0s/it 5:042.7s
                   all       1643       2335      0.722      0.547      0.646      0.486
                 Glass         83        126      0.649      0.492      0.565      0.382
                 Metal        202        244      0.755      0.639      0.728      0.588
                 Ot

## 6. Lưu và Nén Kết Quả (Download)

Nén 2 thư mục kết quả lại thành file `.zip` lưu ở thư mục gốc `/kaggle/working/` để bạn có thể tải về máy một cách dễ dàng.

In [12]:
%cd /kaggle/working/waste-detection2-Stage
!zip -r /kaggle/working/test_evaluation_results.zip runs/detect/results/runs/*test_eval*

print("\n======================================================")
print("Đã nén xong! Hãy tải file test_evaluation_results.zip ở mục Output phía bên phải màn hình.")
print("======================================================")

/kaggle/working/waste-detection2-Stage
  adding: runs/detect/results/runs/baseline_test_eval/ (stored 0%)
  adding: runs/detect/results/runs/baseline_test_eval/val_batch0_pred.jpg (deflated 5%)
  adding: runs/detect/results/runs/baseline_test_eval/BoxPR_curve.png (deflated 10%)
  adding: runs/detect/results/runs/baseline_test_eval/val_batch1_pred.jpg (deflated 4%)
  adding: runs/detect/results/runs/baseline_test_eval/val_batch0_labels.jpg (deflated 5%)
  adding: runs/detect/results/runs/baseline_test_eval/BoxF1_curve.png (deflated 9%)
  adding: runs/detect/results/runs/baseline_test_eval/BoxP_curve.png (deflated 8%)
  adding: runs/detect/results/runs/baseline_test_eval/val_batch2_labels.jpg (deflated 4%)
  adding: runs/detect/results/runs/baseline_test_eval/val_batch1_labels.jpg (deflated 4%)
  adding: runs/detect/results/runs/baseline_test_eval/val_batch2_pred.jpg (deflated 4%)
  adding: runs/detect/results/runs/baseline_test_eval/BoxR_curve.png (deflated 9%)
  adding: runs/detect/res